# S4 — CMIP6 spatial relationship classification / CMIP6空间关系分类

This notebook classifies climatological P, ET, and R relationships on land cells. Analysis is organised **by variable pair** (P → R, then ET → R, then P → ET), not by one composite figure per model.

For each pair, a summary table lists every model × domain. The main scatter overlays **all models that have data** in that domain; **colour encodes the model**, not the climate zone. Zones only choose which panel a point belongs to. Models with no cells in a domain are omitted.

Raw figures and tables are shown first; trimmed versions follow as a sensitivity check.

**中文说明：** 按变量对分析。总表列出所有模型；总散点把有数据的模型画在同一张图上，**一种颜色一个模型，与分区无关**。分区只决定进哪一格。字号加大。先全部 raw，再 trimmed。

Per-model 3×7 figures are still produced after the pair overlays.


## 1. Configuration / 配置

Locate S3 zone climatology output and explicitly load the case-local classifier module.

**中文说明：** 定位S3分区气候态输出，并显式加载case/caseA目录中的classifier.py；后续修改本地分类阈值即可直接作用于S4。

In [1]:
from __future__ import annotations

import importlib
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore', category=FutureWarning)


def locate_case_dir() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / 'cmip_utils.py').exists() and candidate.name == 'caseA':
            return candidate
        nested = candidate / 'case' / 'caseA'
        if (nested / 'cmip_utils.py').exists():
            return nested
    raise FileNotFoundError('Could not locate case/caseA/cmip_utils.py')


CASE_DIR = locate_case_dir()
DATA_ROOT = Path('/Volumes/mimi-T9/CMIP6')
BRANCH_OUTPUT_DIR = CASE_DIR / 'output' / 'S4_branch'
BRANCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(CASE_DIR) not in sys.path:
    sys.path.insert(0, str(CASE_DIR))

# Reload branch-detector dependencies in dependency order.  Without this,
# rerunning the notebook in an existing kernel can mix the new decision tree
# with an older cached KDE peak/track implementation.
for _module_name in (
    'test_kde_peak_valley_score',
    'test_two_condition_kde',
    'test_2d_kde_conditional_tracks',
):
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])

CLASSIFIER_PATH = CASE_DIR / 'classifier0825-branch.py'
if not CLASSIFIER_PATH.exists():
    raise FileNotFoundError(f'Case-local classifier not found: {CLASSIFIER_PATH}')
classifier_spec = importlib.util.spec_from_file_location(
    'caseA_classifier', CLASSIFIER_PATH
)
if classifier_spec is None or classifier_spec.loader is None:
    raise ImportError(f'Cannot create import spec for {CLASSIFIER_PATH}')
classifier_module = importlib.util.module_from_spec(classifier_spec)
classifier_spec.loader.exec_module(classifier_module)
classify_relationship = classifier_module.classify_relationship
CLASSIFIER_DEFAULTS = classifier_module.DEFAULTS

MODELS = [
    'CESM2', 'CNRM-CM6-1', 'CanESM5', 'GFDL-CM4', 'CMCC-CM2-SR5',
]
EXPERIMENTS = ['historical']
START_YEAR = 1985
END_YEAR = 2014
WRITE_OUTPUTS = True
TRIM_LOWER_QUANTILE = 0.01
TRIM_UPPER_QUANTILE = 0.99

VARIABLE_PAIRS = [
    ('P',          'Q', 'P → Q'),
    ('ET',         'Q', 'ET → Q'),
    ('hfls',       'Q', 'hfls → Q'),
    ('hfss',       'Q', 'hfss → Q'),
    ('tran',       'Q', 'tran → Q'),
    ('evspsblsoi', 'Q', 'evspsblsoi → Q'),
    ('mrros',      'Q', 'mrros → Q'),
    ('mrso',       'Q', 'mrso → Q'),
    ('mrsos',      'Q', 'mrsos → Q'),
    ('lai',        'Q', 'lai → Q'),
    ('tas',        'Q', 'tas → Q'),
    ('prsn',       'Q', 'prsn → Q'),
    ('rlds',       'Q', 'rlds → Q'),
    ('rlus',       'Q', 'rlus → Q'),
    ('rsds',       'Q', 'rsds → Q'),
    ('rsus',       'Q', 'rsus → Q'),
]

DOMAINS = [
    ('all_land', 'All land', None),
    ('WW', 'Wet–warm', lambda t: t['analysis_zone'] == 'WW'),
    ('WD', 'Dry–warm', lambda t: t['analysis_zone'] == 'WD'),
    ('CW', 'Wet–cold', lambda t: t['analysis_zone'] == 'CW'),
    ('CD', 'Dry–cold', lambda t: t['analysis_zone'] == 'CD'),
    ('LI', 'Land ice', lambda t: t['analysis_zone'] == 'LI'),
]

# Colours sampled from the climate-region reference map.
ZONE_COLORS = {
    'all_land': '#6B7280',
    'non_ice_land': '#6B7280',
    'WW': '#008070',  # wet-warm, dark teal
    'WD': '#A06018',  # dry-warm, rust brown
    'CW': '#80C8C0',  # wet-cold, mint
    'CD': '#D8C078',  # dry-cold, beige
    'LI': '#3B82F6',  # land ice, blue points
}
ZONE_LEGEND = [
    ('WW', 'Wet–warm'),
    ('CW', 'Wet–cold'),
    ('CD', 'Dry–cold'),
    ('WD', 'Dry–warm'),
    ('LI', 'Land ice'),
]

# Panel background by classification family.
# Simple path = warm tints; complex path = cool tints.
CLASS_FACECOLORS = {
    # Simple path: warm hues
    'Linear': '#FFE8D2',
    'Near-linear': '#F5F0E6',
    'Saturation': '#F6D7C8',
    'Acceleration': '#F7E7A8',
    # Branch path: cool blue
    'Branch': '#D7E8F6',
    'Candidate branch': '#FFF1B8',
    'Two-band': '#E8DDF8',
    # Complex path: teal / grey
    'Complex': '#CDEEEE',
    'No simple': '#E8E8E8',
    'Uncertain': '#E8E8E8',
    'No Global Relationship': '#FFFFFF',
    'No global relationship': '#FFFFFF',
    'No result': '#FFFFFF',
    'Insufficient data': '#F4F4F4',
}

# One stable colour per model on the multi-model overlay scatters.
MODEL_COLOR_CYCLE = [
    '#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00',
    '#56B4E9', '#882255', '#44AA99', '#332288', '#117733',
    '#AA4499', '#88CCEE', '#999933', '#DDCC77', '#000000',
]

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 200,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'axes.edgecolor': '#30343B',
    'axes.linewidth': 1.0, 'axes.titlesize': 14,
    'axes.titleweight': 'semibold', 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'legend.fontsize': 11, 'figure.titlesize': 17,
})

print(f'Case directory: {CASE_DIR}')
print(f'Classifier path:     {CLASSIFIER_PATH}')
print(f'Classifier defaults: {CLASSIFIER_DEFAULTS}')
print(f'Domains: {[d[0] for d in DOMAINS]}')
print(f'Variable pairs: {[v[2] for v in VARIABLE_PAIRS]}')


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Case directory: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA
Classifier path:     /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/classifier0825-branch.py
Classifier defaults: {'gate1_metric': 'mic', 'mic_strong': 0.8, 'mic_intermediate': 0.2, 'dcor_strong': 0.8, 'dcor_intermediate': 0.4, 'branch_method': 'scatter', 'branch_coverage_mode': 'x_range', 'detect_behind_gate': False, 'pearson_threshold': 0.7, 'r2_power_threshold': 0.5, 'linear_b_tolerance': 0.2, 'curvature_b_threshold': 0.5, 'autocorr_threshold': 0.7, 'use_autocorr_filter': False, 'r2_autocorr_shortcut': False, 'shortcut_autocorr_max': 0.7, 'sizer_n_bandwidths': 4, 'sizer_n_grid': 200, 'sizer_h_min': 0.05, 'sizer_h_max': 0.3, 'sizer_interior': (0.05, 0.95), 'sizer_min_run_frac': 0.05, 'sizer_flat_frac': 0.1, 'sizer_flat_ref': 'max', 'tp_support_min': 2, 'force_complex': False}
Domains: ['all_land', 'WW', 'WD', 'CW', 'CD', 'LI']
Variable pairs: ['P → Q', 'ET → Q', 'hfls → Q', 

## 2. Load S3 zone climatology / 加载S3分区气候态

Read the zone climatology table produced by S3. Verify expected columns and zone labels.

**中文说明：** 读取S3输出的分区气候态表，验证列名和分区标签。

In [2]:
RUNS = []
SKIPPED_MODELS = []
for model in MODELS:
    for experiment in EXPERIMENTS:
        search_root = DATA_ROOT / model / experiment
        zone_name = f'zone_climatology_{START_YEAR}_{END_YEAR}.parquet'
        zone_paths = sorted(search_root.glob(f'*/*/land/zones/{zone_name}'))
        if not zone_paths:
            SKIPPED_MODELS.append(model)
            print(f'{model}: no S3 zone climatology, skip')
            continue
        for zp in zone_paths:
            zones_dir = zp.parent
            run_root = zones_dir.parent.parent
            member = run_root.parent.name
            grid = run_root.name
            RUNS.append({
                'model': model,
                'experiment': experiment,
                'member': member,
                'grid': grid,
                'zone_climatology_path': zp,
                'zones_dir': zones_dir,
                'run_root': run_root,
            })

for run in RUNS:
    t = pd.read_parquet(run['zone_climatology_path'])
    t.rename(columns={'R': 'Q'}, inplace=True)
    run['clim'] = t
    required = {'P', 'ET', 'Q', 'analysis_zone', 'surface_class', 'core_zone_cell', 'grid_id', 'land_area'}
    missing = required - set(t.columns)
    if missing:
        raise KeyError(f'Missing columns in S3 output: {sorted(missing)}')
    print(f"{run['model']} {run['member']} {run['grid']}: {len(t)} rows")
    print(f"  analysis_zone: {sorted(t['analysis_zone'].unique())}")
    print(f"  surface_class: {sorted(t['surface_class'].unique())}")

print(f'\nLoaded {len(RUNS)} runs, skipped {len(SKIPPED_MODELS)}: {SKIPPED_MODELS}')
if not RUNS:
    raise FileNotFoundError('No S3 zone climatology found for any requested model')


CESM2 r1i1p1f1 gn: 21013 rows
  analysis_zone: ['CD', 'CW', 'EF', 'KG_unclassified', 'LI', 'WD', 'WW']
  surface_class: ['land_ice', 'non_ice_land']
CNRM-CM6-1 r1i1p1f2 gr: 13639 rows
  analysis_zone: ['CD', 'CW', 'KG_unclassified', 'LI', 'WD', 'WW']
  surface_class: ['land_ice', 'non_ice_land']
CanESM5 r1i1p1f1 gn: 3464 rows
  analysis_zone: ['CD', 'CW', 'KG_unclassified', 'LI', 'WD', 'WW']
  surface_class: ['land_ice', 'non_ice_land']
GFDL-CM4 r1i1p1f1 gr1: 22514 rows
  analysis_zone: ['CD', 'CW', 'EF', 'KG_unclassified', 'LI', 'WD', 'WW']
  surface_class: ['land_ice', 'non_ice_land']
CMCC-CM2-SR5 r1i1p1f1 gn: 20675 rows
  analysis_zone: ['CD', 'CW', 'EF', 'KG_unclassified', 'LI', 'WD', 'WW']
  surface_class: ['land_ice', 'non_ice_land']

Loaded 5 runs, skipped 0: []


## 3. Run classifier on all domain × variable-pair combinations / 对所有分域×变量组合运行分类器

Classify every domain × sample × pair for both raw and trimmed cells. Computation still produces both versions in one pass so later comparisons stay aligned; **display of tables and figures is split**: raw first, trimmed afterwards.

**中文说明：** 分类计算仍一次跑完raw和trimmed，但后面的表和图先全部展示raw，再展示trimmed。


In [3]:
import os
import time as _time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Reload the branch-detector stack before the decision tree so edits apply
# without restarting the kernel.
for _module_name in (
    'test_kde_peak_valley_score',
    'test_two_condition_kde',
    'test_2d_kde_conditional_tracks',
):
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])
classifier_spec = importlib.util.spec_from_file_location(
    'caseA_classifier', CLASSIFIER_PATH
)
classifier_module = importlib.util.module_from_spec(classifier_spec)
classifier_spec.loader.exec_module(classifier_module)
classify_relationship = classifier_module.classify_relationship
CLASSIFIER_DEFAULTS = classifier_module.DEFAULTS
print(f'Reloaded classifier: {CLASSIFIER_PATH}')
print(f'Defaults: {CLASSIFIER_DEFAULTS}')


def prepare_pair_versions(x, y, groups=None, bounds=None):
    """Return finite raw values and a paired 1–99% trimmed sample.

    If *bounds* is supplied, use those all-land bounds.  This keeps every
    plotted trimmed panel identical to the data passed to the classifier.
    """
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    x_raw = x_arr[finite]
    y_raw = y_arr[finite]
    groups_raw = None if groups is None else np.asarray(groups)[finite]
    if len(x_raw) == 0:
        used_bounds = (np.nan, np.nan, np.nan, np.nan)
        keep = np.zeros(0, dtype=bool)
    elif bounds is None:
        x_lo, x_hi = np.quantile(
            x_raw, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE]
        )
        y_lo, y_hi = np.quantile(
            y_raw, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE]
        )
        used_bounds = (float(x_lo), float(x_hi), float(y_lo), float(y_hi))
        keep = (
            (x_raw >= x_lo) & (x_raw <= x_hi)
            & (y_raw >= y_lo) & (y_raw <= y_hi)
        )
    else:
        x_lo, x_hi, y_lo, y_hi = map(float, bounds)
        used_bounds = (x_lo, x_hi, y_lo, y_hi)
        keep = (
            (x_raw >= x_lo) & (x_raw <= x_hi)
            & (y_raw >= y_lo) & (y_raw <= y_hi)
        )
    return {
        'raw': (x_raw, y_raw),
        'trimmed': (x_raw[keep], y_raw[keep]),
        'raw_groups': groups_raw,
        'trimmed_groups': None if groups_raw is None else groups_raw[keep],
        'trim_keep': keep,
        'bounds': used_bounds,
    }


def run_classification(x_clean, y_clean, x_name, y_name, domain_name,
                       sample_name, outlier_version, raw_n_cells, bounds):
    """Run one raw or trimmed classifier and retain trimming diagnostics."""
    x_clean = np.asarray(x_clean, dtype=float)
    y_clean = np.asarray(y_clean, dtype=float)
    n = len(x_clean)
    x_lo, x_hi, y_lo, y_hi = bounds
    n_removed = int(raw_n_cells - n)
    metadata = {
        'domain': domain_name,
        'sample': sample_name,
        'outlier_version': outlier_version,
        'x_var': x_name,
        'y_var': y_name,
        'pair': f'{x_name} → {y_name}',
        'n_cells': n,
        'raw_n_cells': int(raw_n_cells),
        'n_removed': n_removed,
        'removed_fraction': (n_removed / raw_n_cells if raw_n_cells else np.nan),
        'x_trim_lower': x_lo,
        'x_trim_upper': x_hi,
        'y_trim_lower': y_lo,
        'y_trim_upper': y_hi,
    }
    pearson_r_descriptive = np.nan
    if n >= 2 and np.ptp(x_clean) > 0 and np.ptp(y_clean) > 0:
        pearson_r_descriptive = float(np.corrcoef(x_clean, y_clean)[0, 1])
    if n < 10:
        return {
            **metadata,
            'pearson_r_descriptive': pearson_r_descriptive,
            'group': 'Insufficient data',
            'mic': np.nan,
            'has_global_relationship': 'N/A',
            'simple_or_complex': 'N/A',
            'branch_structure': 'N/A',
            'tp_number': None,
        }
    result = classify_relationship(x_clean, y_clean)
    return {
        **metadata,
        **result,
        'pearson_r_descriptive': pearson_r_descriptive,
    }


def run_pair_versions(frame, x_name, y_name, domain_name, sample_name):
    prepared = prepare_pair_versions(frame[x_name], frame[y_name])
    raw_n = len(prepared['raw'][0])
    results = []
    for version in ['raw', 'trimmed']:
        x_values, y_values = prepared[version]
        results.append(run_classification(
            x_values, y_values, x_name, y_name, domain_name, sample_name,
            version, raw_n, prepared['bounds'],
        ))
    return results


# ── Pre-build all jobs as numpy arrays ──────────────────────
# Trimming is computed GLOBALLY (all_land) per model×pair, then applied
# to each domain subset, so all zones share the same outlier thresholds.
JOBS = []
for run in RUNS:
    t = run['clim']
    run_meta = {
        'model': run['model'],
        'experiment': run['experiment'],
        'member': run['member'],
        'grid': run['grid'],
    }

    # Pre-compute global trim bounds per variable pair
    global_bounds = {}
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        if x_var not in t.columns or y_var not in t.columns:
            continue
        x_all = np.asarray(t[x_var], dtype=float)
        y_all = np.asarray(t[y_var], dtype=float)
        finite = np.isfinite(x_all) & np.isfinite(y_all)
        xf, yf = x_all[finite], y_all[finite]
        if len(xf) == 0:
            global_bounds[(x_var, y_var)] = (np.nan, np.nan, np.nan, np.nan)
        else:
            x_lo, x_hi = np.quantile(xf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
            y_lo, y_hi = np.quantile(yf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
            global_bounds[(x_var, y_var)] = (float(x_lo), float(x_hi), float(y_lo), float(y_hi))

    run['global_trim_bounds'] = global_bounds

    for domain_key, domain_label, domain_filter in DOMAINS:
        subset = t.loc[domain_filter(t)] if domain_filter is not None else t
        sample_frames = [('all', subset)]
        if domain_key in ('WW', 'WD', 'CW', 'CD'):
            sample_frames.append(('core', subset.loc[subset['core_zone_cell']]))
        for sample_name, sample_frame in sample_frames:
            for x_var, y_var, pair_label in VARIABLE_PAIRS:
                if (x_var, y_var) not in global_bounds:
                    continue
                x_arr = np.asarray(sample_frame[x_var], dtype=float)
                y_arr = np.asarray(sample_frame[y_var], dtype=float)
                finite = np.isfinite(x_arr) & np.isfinite(y_arr)
                x_raw = x_arr[finite]
                y_raw = y_arr[finite]
                raw_n = len(x_raw)
                bounds = global_bounds[(x_var, y_var)]
                x_lo, x_hi, y_lo, y_hi = bounds
                keep = (
                    (x_raw >= x_lo) & (x_raw <= x_hi)
                    & (y_raw >= y_lo) & (y_raw <= y_hi)
                )
                for version in ['raw', 'trimmed']:
                    if version == 'trimmed':
                        xv, yv = x_raw[keep], y_raw[keep]
                    else:
                        xv, yv = x_raw, y_raw
                    JOBS.append((
                        xv.copy(), yv.copy(),
                        x_var, y_var, domain_key, sample_name,
                        version, raw_n, bounds, run_meta,
                    ))

def _worker(job):
    x, y, x_name, y_name, domain, sample, version, raw_n, bounds, meta = job
    res = run_classification(x, y, x_name, y_name, domain, sample, version, raw_n, bounds)
    res.update(meta)
    return res

# ── Run in parallel ─────────────────────────────────────────
N_WORKERS = max(1, (os.cpu_count() or 4) - 1)
print(f'Running {len(JOBS)} classifications on {N_WORKERS} threads ...')
_t0 = _time.perf_counter()

ALL_RESULTS = []
_done = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = [pool.submit(_worker, job) for job in JOBS]
    for fut in as_completed(futures):
        ALL_RESULTS.append(fut.result())
        _done += 1
        if _done % 50 == 0 or _done == len(JOBS):
            print(f'  {_done}/{len(JOBS)} done')

_elapsed = _time.perf_counter() - _t0
print(f'Finished in {_elapsed:.1f}s')

ALL_RESULTS.sort(key=lambda r: (
    r['model'], r['domain'], r['pair'], r['sample'], r['outlier_version']
))

RESULTS_DF = pd.DataFrame(ALL_RESULTS)

# Audit that every sufficient-data case followed the loaded 0825 branch tree.
_sufficient = RESULTS_DF['n_cells'].ge(10)
_tree_trace_missing = _sufficient & RESULTS_DF['path_trace'].isna()
_branch_gate_open = _sufficient & RESULTS_DF['gate1_gate'].isin(
    ['strong_global', 'intermediate_global']
)
_branch_result_missing = _branch_gate_open & RESULTS_DF['branch_status'].isna()
if _tree_trace_missing.any() or _branch_result_missing.any():
    raise RuntimeError(
        'Decision-tree audit failed: '
        f'missing path_trace={int(_tree_trace_missing.sum())}, '
        f'missing branch result={int(_branch_result_missing.sum())}'
    )
print(
    'Decision-tree audit passed: '
    f'{int(_sufficient.sum())} sufficient-data cases traced; '
    f'{int(_branch_gate_open.sum())} entered the modified branch gate; '
    f'{int((~_sufficient).sum())} insufficient-data cases bypassed.'
)
print(f'\nTotal classifications: {len(RESULTS_DF)}')
for version in ['raw', 'trimmed']:
    n = int((RESULTS_DF['outlier_version'] == version).sum())
    print(f'  {version}: {n} cases')

Reloaded classifier: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/classifier0825-branch.py
Defaults: {'gate1_metric': 'mic', 'mic_strong': 0.8, 'mic_intermediate': 0.2, 'dcor_strong': 0.8, 'dcor_intermediate': 0.4, 'branch_method': 'scatter', 'branch_coverage_mode': 'x_range', 'detect_behind_gate': False, 'pearson_threshold': 0.7, 'r2_power_threshold': 0.5, 'linear_b_tolerance': 0.2, 'curvature_b_threshold': 0.5, 'autocorr_threshold': 0.7, 'use_autocorr_filter': False, 'r2_autocorr_shortcut': False, 'shortcut_autocorr_max': 0.7, 'sizer_n_bandwidths': 4, 'sizer_n_grid': 200, 'sizer_h_min': 0.05, 'sizer_h_max': 0.3, 'sizer_interior': (0.05, 0.95), 'sizer_min_run_frac': 0.05, 'sizer_flat_frac': 0.1, 'sizer_flat_ref': 'max', 'tp_support_min': 2, 'force_complex': False}
Running 1600 classifications on 9 threads ...
  50/1600 done
  100/1600 done
  150/1600 done
  200/1600 done
  250/1600 done
  300/1600 done
  350/1600 done
  400/1600 done
  450/1600 done
  500/160

## 4. Gate-by-gate diagnostic tables / 逐Gate诊断表

Show **all raw tables first**, then the same gates for trimmed. Within each version, diagnose P → R, then ET → R, then P → ET.

**中文说明：** 先把raw的全部Gate表看完，再看trimmed。每套内部仍按P→R、ET→R、P→ET的顺序。


In [4]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# def show_gate_table(title, frame, columns, round_columns=()):
#     table = frame.reindex(columns=columns).copy()
#     for col in round_columns:
#         table[col] = pd.to_numeric(table[col], errors='coerce').round(4)
#     table = table.astype(object).where(pd.notna(table), '-').replace('N/A', '-')
#     print(f'\n=== {title} ({len(table)} cases) ===')
#     display(table.style.set_properties(**{'text-align': 'left', 'white-space': 'nowrap'}))
#     return table
# 
# # Helpers used inside each pair-specific diagnostic block.
# def format_tp_counts(value):
#     if isinstance(value, (list, tuple, np.ndarray)) and len(value) > 0:
#         return '[' + ', '.join(str(int(v)) for v in value) + ']'
#     return '-'
# 
# def sizer_route(tp):
#     if pd.isna(tp):
#         return 'Uncertain (no consensus)'
#     tp = int(tp)
#     if tp == 0:
#         return 'Enter Transition gate'
#     if tp == 1:
#         return 'U-shape'
#     if tp == 2:
#         return 'Cubic'
#     return 'Oscillation'
# 
# PAIR_DIAGNOSTIC_TABLES = {}
# R2_POWER_THRESHOLD = CLASSIFIER_DEFAULTS['r2_power_threshold']
# LINEAR_B_TOLERANCE = CLASSIFIER_DEFAULTS['linear_b_tolerance']
# CURVATURE_B_THRESHOLD = CLASSIFIER_DEFAULTS['curvature_b_threshold']
# 
# for version in ['raw', 'trimmed']:
#     print('\n' + '#' * 88)
#     print(f'{version.upper()} GATE DIAGNOSTICS')
#     print('#' * 88)
#     PAIR_DIAGNOSTIC_TABLES[version] = {}
#     for _, _, pair_label in VARIABLE_PAIRS:
#         pair_cases = RESULTS_DF.loc[
#             (RESULTS_DF['pair'] == pair_label)
#             & (RESULTS_DF['outlier_version'] == version)
#         ].copy()
#         print('\n' + '=' * 88)
#         print(f'{version.upper()} · PAIR DIAGNOSTICS: {pair_label}')
#         print('=' * 88)
# 
#         # Gate 1: every case for this pair starts here.
#         gate1_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Gate 1 — MIC global relationship',
#             pair_cases,
#             ['domain', 'sample', 'outlier_version', 'n_cells',
#              'n_removed', 'removed_fraction', 'mic', 'mic_gate'],
#             ['removed_fraction', 'mic'],
#         )
# 
#         # Gate 2: only cases for this pair that passed Gate 1.
#         gate2_cases = pair_cases.loc[
#             pair_cases['has_global_relationship'] == 'Yes'
#         ].copy()
#         gate2_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Gate 2 — Pearson Simple or Complex',
#             gate2_cases,
#             ['domain', 'sample', 'outlier_version', 'mic', 'pearson_r',
#              'abs_pearson_r', 'simple_or_complex'],
#             ['mic', 'pearson_r', 'abs_pearson_r'],
#         )
# 
#         # Simple step 1: power-law adequacy is always judged by R² first.
#         simple_cases = pair_cases.loc[
#             pair_cases['simple_or_complex'] == 'Simple'
#         ].copy()
#         simple_cases['r2_threshold'] = R2_POWER_THRESHOLD
#         simple_cases['r2_decision'] = np.select(
#             [simple_cases['power_r2'].isna(),
#              simple_cases['power_r2'].ge(R2_POWER_THRESHOLD)],
#             ['Fit failed', 'Pass'],
#             default='Fail',
#         )
#         simple_cases['next_route'] = np.where(
#             simple_cases['r2_decision'].eq('Pass'),
#             'Enter Linear gate', 'Enter Complex',
#         )
#         power_r2_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Simple step 1 — Power-law R² adequacy',
#             simple_cases,
#             ['domain', 'sample', 'outlier_version', 'power_r2', 'r2_threshold',
#              'r2_decision', 'next_route'],
#             ['power_r2', 'r2_threshold'],
#         )
# 
#         # Simple step 2: only R²-pass cases may use |b-1| to test linearity.
#         r2_pass_cases = simple_cases.loc[
#             simple_cases['r2_decision'].eq('Pass')
#         ].copy()
#         r2_pass_cases['linear_threshold'] = LINEAR_B_TOLERANCE
#         r2_pass_cases['linear_decision'] = np.where(
#             r2_pass_cases['abs_b_minus_1'].le(LINEAR_B_TOLERANCE),
#             'Linear', 'Not linear',
#         )
#         r2_pass_cases['next_route'] = np.where(
#             r2_pass_cases['linear_decision'].eq('Linear'),
#             'Final: Linear', 'Enter Curvature magnitude gate',
#         )
#         linear_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Simple step 2 — Linear |b-1|',
#             r2_pass_cases,
#             ['domain', 'sample', 'outlier_version', 'power_b', 'abs_b_minus_1',
#              'linear_threshold', 'linear_decision', 'next_route'],
#             ['power_b', 'abs_b_minus_1', 'linear_threshold'],
#         )
# 
#         # Simple step 3: only R²-pass, non-linear cases test curvature magnitude.
#         nonlinear_cases = r2_pass_cases.loc[
#             r2_pass_cases['linear_decision'].eq('Not linear')
#         ].copy()
#         nonlinear_cases['curvature_threshold'] = CURVATURE_B_THRESHOLD
#         nonlinear_cases['curvature_decision'] = np.where(
#             nonlinear_cases['abs_b_minus_1'].ge(CURVATURE_B_THRESHOLD),
#             'Pass', 'Fail (curvature gap)',
#         )
#         nonlinear_cases['next_route'] = np.where(
#             nonlinear_cases['curvature_decision'].eq('Pass'),
#             'Enter Curvature sign gate', 'Enter Complex',
#         )
#         curvature_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Simple step 3 — Curvature magnitude |b-1|',
#             nonlinear_cases,
#             ['domain', 'sample', 'outlier_version', 'power_b', 'abs_b_minus_1',
#              'curvature_threshold', 'curvature_decision', 'next_route'],
#             ['power_b', 'abs_b_minus_1', 'curvature_threshold'],
#         )
# 
#         # Simple step 4: only curvature-pass cases use the sign of b-1.
#         curvature_pass_cases = nonlinear_cases.loc[
#             nonlinear_cases['curvature_decision'].eq('Pass')
#         ].copy()
#         curvature_pass_cases['b_minus_1'] = (
#             curvature_pass_cases['power_b'] - 1
#         )
#         curvature_pass_cases['sign_decision'] = np.where(
#             curvature_pass_cases['b_minus_1'].gt(0),
#             'b-1 > 0', 'b-1 < 0',
#         )
#         curvature_pass_cases['final_group'] = curvature_pass_cases['power_decision']
#         curvature_sign_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Simple step 4 — Curvature sign',
#             curvature_pass_cases,
#             ['domain', 'sample', 'outlier_version', 'power_b', 'b_minus_1',
#              'sign_decision', 'final_group'],
#             ['power_b', 'b_minus_1'],
#         )
# 
#         # Complex step 1: direct Complex plus Simple fallback for this pair.
#         complex_cases = pair_cases.loc[
#             pair_cases['complex_path_used'].eq(True)
#         ].copy()
#         complex_cases['entered_from'] = np.where(
#             complex_cases['simple_or_complex'].eq('Simple'),
#             'Simple fallback', 'Gate 2 Complex',
#         )
#         branch_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Complex step 1 — Branch detection',
#             complex_cases,
#             ['domain', 'sample', 'outlier_version', 'entered_from',
#              'branch_fraction', 'branch_structure'],
#             ['branch_fraction'],
#         )
# 
#         # Complex step 2: only non-branch cases for this pair.
#         sizer_cases = complex_cases.loc[
#             complex_cases['branch_structure'] == 'No'
#         ].copy()
#         sizer_cases['tp_counts'] = (
#             sizer_cases['sizer_tp_counts'].map(format_tp_counts)
#         )
#         sizer_cases['sizer_result'] = sizer_cases['tp_number'].map(sizer_route)
#         sizer_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Complex step 2 — SiZer turning points',
#             sizer_cases,
#             ['domain', 'sample', 'outlier_version',
#              'tp_counts', 'tp_number', 'sizer_result'],
#         )
# 
#         # Complex step 3: only TP=0 cases for this pair.
#         transition_cases = sizer_cases.loc[
#             sizer_cases['tp_number'].eq(0)
#         ].copy()
#         transition_table = show_gate_table(
#             f'{version.upper()} · {pair_label} · Complex step 3 — Transition dip test',
#             transition_cases,
#             ['domain', 'sample', 'outlier_version', 'dip_pvalue',
#              'transition_detected', 'group'],
#             ['dip_pvalue'],
#         )
# 
#         PAIR_DIAGNOSTIC_TABLES[version][pair_label] = {
#             'gate1': gate1_table,
#             'gate2': gate2_table,
#             'power_r2': power_r2_table,
#             'linear': linear_table,
#             'curvature': curvature_table,
#             'curvature_sign': curvature_sign_table,
#             'branch': branch_table,
#             'sizer': sizer_table,
#             'transition': transition_table,
#         }
# 

### 4b. Final classification summary / 最终分类汇总

Show one compact final table per variable pair, **raw first, then trimmed**.

**中文说明：** 每个pair先给raw的最终类别表，再给trimmed。


In [5]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# PAIR_FINAL_TABLES = {}
# for version in ['raw', 'trimmed']:
#     print('\n' + '#' * 88)
#     print(f'{version.upper()} FINAL CLASSIFICATION')
#     print('#' * 88)
#     PAIR_FINAL_TABLES[version] = {}
#     for _, _, pair_label in VARIABLE_PAIRS:
#         final_summary = RESULTS_DF.loc[
#             (RESULTS_DF['pair'] == pair_label)
#             & (RESULTS_DF['outlier_version'] == version),
#             ['model', 'domain', 'sample', 'n_cells',
#              'n_removed', 'removed_fraction', 'group', 'path_trace'],
#         ].copy()
#         final_display = final_summary.astype(object).where(
#             pd.notna(final_summary), '-'
#         ).replace('N/A', '-')
#         print(f'\n=== {version.upper()} · {pair_label} · Final classification ({len(final_display)} cases) ===')
#         display(
#             final_display.style.set_properties(
#                 **{'text-align': 'left', 'white-space': 'nowrap'}
#             )
#         )
#         PAIR_FINAL_TABLES[version][pair_label] = final_display
# 

## 6. Pair analysis — RAW / 按变量对（raw）

For each variable pair: one summary table (same columns as the classification tables), then one multi-model figure. Each panel is a domain; every model with data is overlaid; **colour = model**.

**中文说明：** 每个变量对先出总表，再出总散点。一格一个域，有数的模型叠在一起，颜色只区分模型。


In [6]:
def style_axes(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#30343B')
        spine.set_linewidth(1.0)
    ax.tick_params(
        direction='out', length=4.0, width=0.9,
        color='#30343B', labelsize=11,
    )
    ax.grid(True, color='#D3D3D3', linewidth=0.6, alpha=0.7, zorder=0)


UNIT_LABELS = {
    'P': 'Precipitation, P (mm yr⁻¹)',
    'ET': 'Evapotranspiration, ET (mm yr⁻¹)',
    'Q': 'Total runoff, Q (mm yr⁻¹)',
    'hfls': 'Latent heat flux (W m⁻²)',
    'hfss': 'Sensible heat flux (W m⁻²)',
    'tran': 'Transpiration (mm yr⁻¹)',
    'evspsblsoi': 'Soil evaporation (mm yr⁻¹)',
    'mrros': 'Surface runoff (mm yr⁻¹)',
    'mrso': 'Total soil moisture (kg m⁻²)',
    'mrsos': 'Topsoil moisture (kg m⁻²)',
    'lai': 'Leaf area index (m² m⁻²)',
    'tas': 'Near-surface temperature (K)',
    'prsn': 'Snowfall (mm yr⁻¹)',
    'rlds': 'Downward LW radiation (W m⁻²)',
    'rlus': 'Upward LW radiation (W m⁻²)',
    'rsds': 'Downward SW radiation (W m⁻²)',
    'rsus': 'Upward SW radiation (W m⁻²)',
}
PAIR_SLUGS = {label: f'{x}_{y}' for x, y, label in VARIABLE_PAIRS}

# Full names for figure titles (abbreviation → readable name)
PAIR_FULL_NAMES = {}
for _x, _y, _label in VARIABLE_PAIRS:
    _x_name = UNIT_LABELS.get(_x, _x).split('(')[0].strip().rstrip(',')
    _y_name = UNIT_LABELS.get(_y, _y).split('(')[0].strip().rstrip(',')
    PAIR_FULL_NAMES[_label] = f'{_x_name} → {_y_name}'
SUMMARY_COLUMNS = [
    'model', 'domain', 'n_cells', 'n_removed', 'removed_fraction',
    'mic', 'pearson_r_descriptive', 'group', 'path_trace',
]
PAIR_FIGURE_DIR = BRANCH_OUTPUT_DIR
PAIR_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def assign_model_colors(model_names):
    ordered = list(dict.fromkeys(model_names))
    return {
        name: MODEL_COLOR_CYCLE[i % len(MODEL_COLOR_CYCLE)]
        for i, name in enumerate(ordered)
    }


MODEL_COLORS = assign_model_colors([run['model'] for run in RUNS])
print('Model colours:')
for name, color in MODEL_COLORS.items():
    print(f'  {name}: {color}')


def padded_limits(values, pad_fraction=0.04):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (-1.0, 1.0)
    lower, upper = float(values.min()), float(values.max())
    span = upper - lower
    if not np.isfinite(span) or span == 0:
        span = max(abs(lower), 1.0)
    pad = pad_fraction * span
    return lower - pad, upper + pad


def lookup_result_row(model, domain_key, x_var, y_var, version):
    rows = RESULTS_DF.loc[
        (RESULTS_DF['model'] == model)
        & (RESULTS_DF['domain'] == domain_key)
        & (RESULTS_DF['sample'] == 'all')
        & (RESULTS_DF['x_var'] == x_var)
        & (RESULTS_DF['y_var'] == y_var)
        & (RESULTS_DF['outlier_version'] == version)
    ]
    if len(rows) != 1:
        return None
    return rows.iloc[0]


def collect_overlay_series(x_var, y_var, domain_key, domain_filter, version):
    """Points for every model that has finite cells in this domain."""
    series = []
    for run in RUNS:
        table = run['clim']
        subset = table if domain_filter is None else table.loc[domain_filter(table)]
        if subset.empty:
            continue
        prepared = prepare_pair_versions(
            subset[x_var], subset[y_var],
            bounds=run['global_trim_bounds'][(x_var, y_var)],
        )
        x_raw, y_raw = prepared['raw']
        if len(x_raw) == 0:
            continue
        x_vals, y_vals = prepared[version]
        keep = prepared['trim_keep']
        series.append({
            'model': run['model'],
            'x': x_vals,
            'y': y_vals,
            'x_raw': x_raw,
            'y_raw': y_raw,
            'trim_keep': keep,
            'xlim_src': x_raw,
            'ylim_src': y_raw,
            'result': lookup_result_row(
                run['model'], domain_key, x_var, y_var, version
            ),
        })
    return series


def pair_summary_table(pair_label, version):
    table = RESULTS_DF.loc[
        (RESULTS_DF['pair'] == pair_label)
        & (RESULTS_DF['outlier_version'] == version)
        & (RESULTS_DF['sample'] == 'all'),
        [c for c in SUMMARY_COLUMNS if c in RESULTS_DF.columns],
    ].copy()
    table = table.sort_values(['domain', 'model'])
    display_table = table.astype(object).where(pd.notna(table), '-').replace('N/A', '-')
    print(
        f'\n=== {version.upper()} · {pair_label} · all models '
        f'({len(display_table)} rows) ==='
    )
    display(
        display_table.style.set_properties(
            **{'text-align': 'left', 'white-space': 'nowrap', 'font-size': '12px'}
        )
    )
    out = PAIR_FIGURE_DIR / f'S4_summary_{PAIR_SLUGS[pair_label]}_{version}.csv'
    if WRITE_OUTPUTS:
        table.to_csv(out, index=False)
        print(f'saved {out}')
    return table


def draw_pair_overlay_figure(x_var, y_var, pair_label, version):
    n_domains = len(DOMAINS)
    n_cols = 4
    n_rows = int(np.ceil(n_domains / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()

    legend_handles = []
    seen_models = []

    for idx, (domain_key, domain_label, domain_filter) in enumerate(DOMAINS):
        ax = axes[idx]
        series = collect_overlay_series(
            x_var, y_var, domain_key, domain_filter, version
        )
        x_pool, y_pool = [], []
        for item in series:
            x_pool.append(item['xlim_src'])
            y_pool.append(item['ylim_src'])
            color = MODEL_COLORS[item['model']]
            if version == 'trimmed':
                dropped = ~np.asarray(item['trim_keep'], dtype=bool)
                if dropped.any():
                    ax.scatter(
                        item['x_raw'][dropped], item['y_raw'][dropped],
                        marker='x', s=10, c=color, alpha=0.55,
                        linewidths=0.7, zorder=1, rasterized=True,
                    )
            if len(item['x']) > 0:
                ax.scatter(
                    item['x'], item['y'],
                    s=5, c=color, alpha=0.14,
                    edgecolors='none', rasterized=True, zorder=2,
                    label=item['model'],
                )
            if item['model'] not in seen_models:
                seen_models.append(item['model'])
                legend_handles.append(
                    Line2D(
                        [0], [0], marker='o', color='none',
                        markerfacecolor=color, markeredgecolor='none',
                        markersize=10, alpha=0.6,
                        label=item['model'],
                    )
                )
        if series:
            ax.set_xlim(padded_limits(np.concatenate(x_pool)))
            ax.set_ylim(padded_limits(np.concatenate(y_pool)))
        else:
            ax.text(
                0.5, 0.5, 'No models with data',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=13, color='#6B7280',
            )
        ax.set_title(f'{domain_label}  ({domain_key})', fontsize=14, pad=8)
        ax.set_xlabel(UNIT_LABELS[x_var], fontsize=13)
        ax.set_ylabel(UNIT_LABELS[y_var], fontsize=13)
        style_axes(ax)
        panel_letter = chr(ord('a') + idx)
        ax.text(
            0.02, 0.98, panel_letter,
            transform=ax.transAxes, va='top', ha='left',
            fontsize=13, fontweight='bold', color='#222222',
        )

    for ax in axes[n_domains:]:
        ax.set_visible(False)

    fig.suptitle(
        f'{version.upper()}  ·  {PAIR_FULL_NAMES.get(pair_label, pair_label)}  ·  all models with data',
        fontsize=18, fontweight='semibold', y=1.05,
    )
    if legend_handles:
        fig.legend(
            handles=legend_handles,
            loc='lower center',
            ncol=min(5, len(legend_handles)),
            bbox_to_anchor=(0.5, -0.14),
            frameon=True,
            fancybox=True,
            framealpha=0.5,
            edgecolor='none',
            fontsize=12,
            markerscale=1.3,
            title='Model (colour)',
            title_fontsize=13,
        )
        if version == 'trimmed':
            fig.legend(
                handles=[
                    Line2D(
                        [0], [0], marker='x', color='#444444',
                        linestyle='none', markersize=8,
                        label='Removed (outside 1st–99th percentile on x or y)',
                    )
                ],
                loc='lower center',
                bbox_to_anchor=(0.5, -0.20),
                frameon=True,
                fancybox=True,
                framealpha=0.5,
                edgecolor='none',
                fontsize=11,
            )
    if WRITE_OUTPUTS:
        path = PAIR_FIGURE_DIR / f'S4_overlay_{PAIR_SLUGS[pair_label]}_{version}.png'
        fig.savefig(path, dpi=180, bbox_inches='tight')
        print(f'saved {path}')
    plt.show()
    plt.close(fig)


print('RAW pair analysis')
print('=' * 72)
for x_var, y_var, pair_label in VARIABLE_PAIRS:
    print('\n' + '#' * 72)
    print(f'RAW · {pair_label}')
    print('#' * 72)
    pair_summary_table(pair_label, 'raw')
    draw_pair_overlay_figure(x_var, y_var, pair_label, 'raw')


Model colours:
  CESM2: #0072B2
  CNRM-CM6-1: #D55E00
  CanESM5: #009E73
  GFDL-CM4: #CC79A7
  CMCC-CM2-SR5: #E69F00
RAW pair analysis

########################################################################
RAW · P → Q
########################################################################

=== RAW · P → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
4,CESM2,CD,2325,0,0.000000,0.618400,0.811673,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
324,CMCC-CM2-SR5,CD,2289,0,0.000000,0.528700,0.851512,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
644,CNRM-CM6-1,CD,1427,0,0.000000,0.688100,0.899163,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
964,CanESM5,CD,373,0,0.000000,0.629800,0.858722,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
1284,GFDL-CM4,CD,2272,0,0.000000,0.376000,0.703118,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
68,CESM2,CW,4642,0,0.000000,0.392000,0.843084,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
388,CMCC-CM2-SR5,CW,4507,0,0.000000,0.427700,0.860085,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
708,CNRM-CM6-1,CW,2821,0,0.000000,0.622400,0.944006,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
1028,CanESM5,CW,806,0,0.000000,0.401300,0.858173,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
1348,GFDL-CM4,CW,4714,0,0.000000,0.226900,0.629647,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_P_Q_raw.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_P_Q_raw.png

########################################################################
RAW · ET → Q
########################################################################

=== RAW · ET → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
0,CESM2,CD,2325,0,0.000000,0.183700,0.017891,No Global Relationship,MIC:no_global -> No Global Relationship
320,CMCC-CM2-SR5,CD,2289,0,0.000000,0.192700,0.205216,No Global Relationship,MIC:no_global -> No Global Relationship
640,CNRM-CM6-1,CD,1427,0,0.000000,0.331600,0.118165,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
960,CanESM5,CD,373,0,0.000000,0.259400,0.077582,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
1280,GFDL-CM4,CD,2272,0,0.000000,0.237900,0.091963,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
64,CESM2,CW,4642,0,0.000000,0.242800,0.093989,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
384,CMCC-CM2-SR5,CW,4507,0,0.000000,0.184800,0.126966,No Global Relationship,MIC:no_global -> No Global Relationship
704,CNRM-CM6-1,CW,2821,0,0.000000,0.155900,0.298301,No Global Relationship,MIC:no_global -> No Global Relationship
1024,CanESM5,CW,806,0,0.000000,0.194800,0.193259,No Global Relationship,MIC:no_global -> No Global Relationship
1344,GFDL-CM4,CW,4714,0,0.000000,0.237000,-0.013573,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_ET_Q_raw.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_ET_Q_raw.png

########################################################################
RAW · hfls → Q
########################################################################

=== RAW · hfls → Q · all models (30 rows) ===


...

saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_rlus_Q_raw.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_rlus_Q_raw.png

########################################################################
RAW · rsds → Q
########################################################################

=== RAW · rsds → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
48,CESM2,CD,2325,0,0.000000,0.338900,-0.370783,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus
368,CMCC-CM2-SR5,CD,2289,0,0.000000,0.420900,-0.283912,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0
688,CNRM-CM6-1,CD,1427,0,0.000000,0.263700,-0.253728,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus
1008,CanESM5,CD,373,0,0.000000,0.489400,-0.385422,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0
1328,GFDL-CM4,CD,2272,0,0.000000,0.355100,-0.334209,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0
112,CESM2,CW,4642,0,0.000000,0.277800,-0.178972,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
432,CMCC-CM2-SR5,CW,4507,0,0.000000,0.221300,-0.100924,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
752,CNRM-CM6-1,CW,2821,0,0.000000,0.156500,0.064020,No Global Relationship,MIC:no_global -> No Global Relationship
1072,CanESM5,CW,806,0,0.000000,0.235700,-0.070783,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus
1392,GFDL-CM4,CW,4714,0,0.000000,0.215000,0.055732,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_rsds_Q_raw.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_rsds_Q_raw.png

########################################################################
RAW · rsus → Q
########################################################################

=== RAW · rsus → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
52,CESM2,CD,2325,0,0.000000,0.312200,-0.263724,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus
372,CMCC-CM2-SR5,CD,2289,0,0.000000,0.242000,-0.274504,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
692,CNRM-CM6-1,CD,1427,0,0.000000,0.169800,-0.014994,No Global Relationship,MIC:no_global -> No Global Relationship
1012,CanESM5,CD,373,0,0.000000,0.323300,-0.168204,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
1332,GFDL-CM4,CD,2272,0,0.000000,0.133100,0.108159,No Global Relationship,MIC:no_global -> No Global Relationship
116,CESM2,CW,4642,0,0.000000,0.141300,-0.084491,No Global Relationship,MIC:no_global -> No Global Relationship
436,CMCC-CM2-SR5,CW,4507,0,0.000000,0.166600,-0.186372,No Global Relationship,MIC:no_global -> No Global Relationship
756,CNRM-CM6-1,CW,2821,0,0.000000,0.152200,-0.102290,No Global Relationship,MIC:no_global -> No Global Relationship
1076,CanESM5,CW,806,0,0.000000,0.201600,-0.186220,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
1396,GFDL-CM4,CW,4714,0,0.000000,0.200800,0.175899,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_rsus_Q_raw.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_rsus_Q_raw.png


## 6b. Pair analysis — TRIMMED / 按变量对（trimmed）

Same tables and multi-model overlays as the raw section, after every raw pair has been shown. Colour is still the model, not the zone.

**中文说明：** 全部 raw 变量对看完后再看 trimmed。颜色仍然只区分模型。

Trimmed panels keep retained cells as circles and mark removed cells with **x**.

**中文说明：** trimmed 图里保留的点仍是圆点，被去掉的点用 x。


In [7]:
print('TRIMMED pair analysis')
print('=' * 72)
for x_var, y_var, pair_label in VARIABLE_PAIRS:
    print('\n' + '#' * 72)
    print(f'TRIMMED · {pair_label}')
    print('#' * 72)
    pair_summary_table(pair_label, 'trimmed')
    draw_pair_overlay_figure(x_var, y_var, pair_label, 'trimmed')


TRIMMED pair analysis

########################################################################
TRIMMED · P → Q
########################################################################

=== TRIMMED · P → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
5,CESM2,CD,2244,81,0.034839,0.605700,0.860186,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
325,CMCC-CM2-SR5,CD,2223,66,0.028834,0.514000,0.823112,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
645,CNRM-CM6-1,CD,1401,26,0.018220,0.680200,0.901239,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
965,CanESM5,CD,371,2,0.005362,0.622400,0.859959,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
1285,GFDL-CM4,CD,2270,2,0.000880,0.375500,0.692054,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0
69,CESM2,CW,4623,19,0.004093,0.388400,0.809486,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
389,CMCC-CM2-SR5,CW,4489,18,0.003994,0.428900,0.837885,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
709,CNRM-CM6-1,CW,2808,13,0.004608,0.620200,0.921449,Near-linear,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Near-linear
1029,CanESM5,CW,805,1,0.001241,0.399700,0.829879,Acceleration,MIC:intermediate_global -> Branch:No -> Pearson:Simple -> Power:Acceleration
1349,GFDL-CM4,CW,4708,6,0.001273,0.225700,0.582755,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_P_Q_trimmed.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_P_Q_trimmed.png

########################################################################
TRIMMED · ET → Q
########################################################################

=== TRIMMED · ET → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
1,CESM2,CD,2246,79,0.033978,0.185500,0.273713,No Global Relationship,MIC:no_global -> No Global Relationship
321,CMCC-CM2-SR5,CD,2223,66,0.028834,0.190300,0.216995,No Global Relationship,MIC:no_global -> No Global Relationship
641,CNRM-CM6-1,CD,1407,20,0.014015,0.318400,0.125279,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
961,CanESM5,CD,373,0,0.000000,0.259400,0.077582,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
1281,GFDL-CM4,CD,2271,1,0.000440,0.237800,0.100540,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
65,CESM2,CW,4619,23,0.004955,0.239900,0.077977,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus
385,CMCC-CM2-SR5,CW,4489,18,0.003994,0.181000,0.119924,No Global Relationship,MIC:no_global -> No Global Relationship
705,CNRM-CM6-1,CW,2808,13,0.004608,0.151800,0.298537,No Global Relationship,MIC:no_global -> No Global Relationship
1025,CanESM5,CW,805,1,0.001241,0.198400,0.169957,No Global Relationship,MIC:no_global -> No Global Relationship
1345,GFDL-CM4,CW,4708,6,0.001273,0.238800,-0.041380,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_ET_Q_trimmed.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_ET_Q_trimmed.png

########################################################################
TRIMMED · hfls → Q
########################################################################

=== TRIMMED · hfls → Q · all models (30 rows) ===


...

saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_rlus_Q_trimmed.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_rlus_Q_trimmed.png

########################################################################
TRIMMED · rsds → Q
########################################################################

=== TRIMMED · rsds → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
49,CESM2,CD,2217,108,0.046452,0.341500,-0.417824,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus
369,CMCC-CM2-SR5,CD,2209,80,0.034950,0.429000,-0.320621,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0
689,CNRM-CM6-1,CD,1388,39,0.027330,0.268300,-0.243824,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
1009,CanESM5,CD,373,0,0.000000,0.489400,-0.385422,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0
1329,GFDL-CM4,CD,2235,37,0.016285,0.353400,-0.393016,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
113,CESM2,CW,4512,130,0.028005,0.281400,-0.170350,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
433,CMCC-CM2-SR5,CW,4368,139,0.030841,0.225100,-0.075564,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
753,CNRM-CM6-1,CW,2777,44,0.015597,0.150600,0.077124,No Global Relationship,MIC:no_global -> No Global Relationship
1073,CanESM5,CW,782,24,0.029777,0.249000,-0.065926,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0
1393,GFDL-CM4,CW,4644,70,0.014849,0.212100,0.028249,Candidate branch,MIC:intermediate_global -> Branch:Candidate


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_rsds_Q_trimmed.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_rsds_Q_trimmed.png

########################################################################
TRIMMED · rsus → Q
########################################################################

=== TRIMMED · rsus → Q · all models (30 rows) ===


,model,domain,n_cells,n_removed,removed_fraction,mic,pearson_r_descriptive,group,path_trace
53,CESM2,CD,2246,79,0.033978,0.302200,-0.414909,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:no_consensus
373,CMCC-CM2-SR5,CD,2218,71,0.031018,0.241500,-0.294514,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:1
693,CNRM-CM6-1,CD,1401,26,0.018220,0.163000,-0.003356,No Global Relationship,MIC:no_global -> No Global Relationship
1013,CanESM5,CD,373,0,0.000000,0.323300,-0.168204,Complex,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:2
1333,GFDL-CM4,CD,2259,13,0.005722,0.132800,0.084696,No Global Relationship,MIC:no_global -> No Global Relationship
117,CESM2,CW,4545,97,0.020896,0.137900,-0.012640,No Global Relationship,MIC:no_global -> No Global Relationship
437,CMCC-CM2-SR5,CW,4422,85,0.018860,0.159300,-0.157408,No Global Relationship,MIC:no_global -> No Global Relationship
757,CNRM-CM6-1,CW,2744,77,0.027295,0.138200,-0.062513,No Global Relationship,MIC:no_global -> No Global Relationship
1077,CanESM5,CW,777,29,0.035980,0.196200,-0.191367,No Global Relationship,MIC:no_global -> No Global Relationship
1397,GFDL-CM4,CW,4646,68,0.014425,0.203600,0.158877,Uncertain,MIC:intermediate_global -> Branch:No -> Pearson:Complex -> Complex:direct -> SiZer:0


saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_summary_rsus_Q_trimmed.csv
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_overlay_rsus_Q_trimmed.png


## 6c. Per-model scatter plots — RAW / 单模型散点（raw）

Keep the original one-model figures: three variable-pair rows × seven domain columns. These sit **after** the multi-model pair overlays, they are not replaced by them. The all-non-ice-land column is coloured by climate zone; land ice is the last column in blue.

**中文说明：** 保留原来的单模型总图（三行变量对 × 七列域）。多模型总图在前面，这里不删。非冰陆地按四区着色，冰盖最后一列蓝点。


In [8]:
N_EQUAL_COUNT_BINS = 10  # 等频分箱数量（用于散点图可视化）


def equal_count_bin_medians(x, y, n_bins=N_EQUAL_COUNT_BINS):
    frame = pd.DataFrame({'x': x, 'y': y}).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    if len(frame) < 2 or frame['x'].nunique() < 2:
        return pd.DataFrame(columns=['x', 'y', 'n'])
    q = min(int(n_bins), int(frame['x'].nunique()))
    frame['bin'] = pd.qcut(frame['x'], q=q, duplicates='drop')
    curve = frame.groupby('bin', observed=True).agg(
        x=('x', 'median'), y=('y', 'median'), n=('y', 'size')
    ).reset_index(drop=True)
    return curve.sort_values('x')


def draw_power_fit(ax, result_row, x_values):
    needed = ['power_a', 'power_b', 'power_c', 'power_r2']
    if any(pd.isna(result_row.get(name, np.nan)) for name in needed):
        return False
    positive_x = np.asarray(x_values, dtype=float)
    positive_x = positive_x[np.isfinite(positive_x) & (positive_x > 1e-6)]
    if len(positive_x) < 2:
        return False
    x_fit = np.linspace(positive_x.min(), positive_x.max(), 350)
    a = float(result_row['power_a'])
    b = float(result_row['power_b'])
    c = float(result_row['power_c'])
    y_fit = a * np.power(np.maximum(x_fit, 1e-10), b) + c
    ax.plot(
        x_fit, y_fit, color='#B12A68', linewidth=1.8,
        linestyle=(0, (5, 3)), zorder=5,
    )
    return True


def metric_text(row, column):
    value = row.get(column, np.nan)
    return f'{value:.3f}' if pd.notna(value) else '—'


def display_class(group):
    return {
        'No Global Relationship': 'No global relationship',
    }.get(str(group), str(group))


def classification_facecolor(group):
    if group is None:
        return '#FFFFFF'
    try:
        if isinstance(group, float) and np.isnan(group):
            return '#FFFFFF'
    except TypeError:
        pass
    key = str(group)
    if key in CLASS_FACECOLORS:
        return CLASS_FACECOLORS[key]
    return CLASS_FACECOLORS.get(display_class(key), '#FFFFFF')


def format_panel_annotation(result_row, version, raw_row=None):
    """Return (class_name, detail_text). Class is drawn bold; no 'Final class' prefix."""
    if result_row is None:
        return 'No result', ''
    final_class = display_class(result_row['group'])
    if final_class in {'', 'nan', 'None'}:
        final_class = 'No result'
    details = (
        f"MIC={metric_text(result_row, 'mic')}  |  "
        f"r={metric_text(result_row, 'pearson_r_descriptive')}\n"
        f"n={int(result_row['n_cells'])}"
    )
    if version == 'trimmed':
        details += (
            f"  |  rem={int(result_row['n_removed'])} "
            f"({100 * float(result_row['removed_fraction']):.1f}%)"
        )
        if raw_row is not None and raw_row['group'] != result_row['group']:
            details += (
                f"\n{display_class(raw_row['group'])} → {final_class}"
            )
    if pd.notna(result_row.get('power_r2', np.nan)):
        details += (
            f"\nPower: R²={float(result_row['power_r2']):.3f}  "
            f"b={float(result_row['power_b']):.3f}"
        )
        if pd.notna(result_row.get('linearity_score', np.nan)):
            details += f"  L={float(result_row['linearity_score']):.2f}"
    return final_class, details


def draw_class_annotation(ax, class_name, details, *, fontsize=9.2, y=0.96):
    """Bold class name, then metrics — one box, no overlapping text."""
    from matplotlib.offsetbox import AnnotationBbox, TextArea, VPacker

    title = TextArea(
        class_name,
        textprops={
            'fontsize': fontsize + 0.8,
            'fontweight': 'bold',
            'color': '#111111',
            'family': 'DejaVu Sans',
        },
    )
    children = [title]
    if details:
        children.append(
            TextArea(
                details,
                textprops={
                    'fontsize': fontsize,
                    'fontweight': 'normal',
                    'color': '#202020',
                    'family': 'DejaVu Sans',
                },
            )
        )
    pack = VPacker(children=children, align='left', pad=0, sep=3)
    box = AnnotationBbox(
        pack,
        (0.03, y),
        xycoords='axes fraction',
        box_alignment=(0.0, 1.0),
        frameon=True,
        pad=0.30,
        bboxprops={
            'boxstyle': 'round,pad=0.35',
            'facecolor': 'white',
            'edgecolor': '#C9C9C9',
            'alpha': 0.92,
        },
        zorder=8,
    )
    ax.add_artist(box)


def scatter_points(ax, x_data, y_data, domain_key, groups,
                   dropped=None, dropped_groups=None):
    if dropped is not None:
        xd, yd = dropped
        if len(xd) > 0:
            if domain_key == 'non_ice_land' and dropped_groups is not None:
                dcolors = [ZONE_COLORS.get(str(z), '#B0B0B0') for z in dropped_groups]
                ax.scatter(
                    xd, yd, marker='x', s=10, c=dcolors, alpha=0.55,
                    linewidths=0.7, zorder=1, rasterized=True,
                )
            else:
                ax.scatter(
                    xd, yd, marker='x', s=10,
                    color=ZONE_COLORS.get(domain_key, '#555555'),
                    alpha=0.55, linewidths=0.6, zorder=1, rasterized=True,
                )
    if len(x_data) == 0:
        if dropped is None or len(dropped[0]) == 0:
            ax.text(
                0.5, 0.5, 'Unavailable', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='#6B7280',
            )
        return
    if domain_key == 'non_ice_land' and groups is not None:
        colors = [ZONE_COLORS.get(str(z), '#B0B0B0') for z in groups]
        ax.scatter(
            x_data, y_data, s=5, c=colors, alpha=0.14,
            edgecolors='none', rasterized=True, zorder=2,
        )
        return
    color = ZONE_COLORS.get(domain_key, '#555555')
    size = 5
    ax.scatter(
        x_data, y_data, s=size, color=color, alpha=0.14,
        edgecolors='none', rasterized=True, zorder=2,
    )


def build_panel_payloads(run):
    t = run['clim']
    payloads = {}
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        for domain_key, domain_label, domain_filter in DOMAINS:
            subset = t.loc[domain_filter(t)] if domain_filter is not None else t
            prepared = prepare_pair_versions(
                subset[x_var], subset[y_var],
                groups=subset['analysis_zone'] if 'analysis_zone' in subset else None,
                bounds=run['global_trim_bounds'][(x_var, y_var)],
            )
            x_raw, y_raw = prepared['raw']
            result_rows = RESULTS_DF.loc[
                (RESULTS_DF['model'] == run['model'])
                & (RESULTS_DF['experiment'] == run['experiment'])
                & (RESULTS_DF['member'] == run['member'])
                & (RESULTS_DF['grid'] == run['grid'])
                & (RESULTS_DF['domain'] == domain_key)
                & (RESULTS_DF['sample'] == 'all')
                & (RESULTS_DF['x_var'] == x_var)
                & (RESULTS_DF['y_var'] == y_var)
            ]
            payloads[(pair_label, domain_key)] = {
                'x_var': x_var,
                'y_var': y_var,
                'pair_label': pair_label,
                'domain_key': domain_key,
                'domain_label': domain_label,
                'prepared': prepared,
                'result_rows': result_rows,
                'xlim': padded_limits(x_raw),
                'ylim': padded_limits(y_raw),
            }
    return payloads


def draw_relationship_figure(run, version, panel_payloads):
    n_rows = len(VARIABLE_PAIRS)
    n_cols = len(DOMAINS)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.9 * n_cols, 4.15 * n_rows),
        constrained_layout=True, squeeze=False,
    )

    for row_i, (x_var, y_var, pair_label) in enumerate(VARIABLE_PAIRS):
        for col_i, (domain_key, domain_label, _) in enumerate(DOMAINS):
            ax = axes[row_i, col_i]
            payload = panel_payloads[(pair_label, domain_key)]
            x_data, y_data = payload['prepared'][version]
            groups = payload['prepared'][f'{version}_groups']
            dropped = None
            dropped_groups = None
            if version == 'trimmed':
                keep = np.asarray(payload['prepared']['trim_keep'], dtype=bool)
                x_raw, y_raw = payload['prepared']['raw']
                dropped = (x_raw[~keep], y_raw[~keep])
                g_raw = payload['prepared']['raw_groups']
                if g_raw is not None:
                    dropped_groups = g_raw[~keep]
            scatter_points(
                ax, x_data, y_data, domain_key, groups,
                dropped=dropped, dropped_groups=dropped_groups,
            )

            bin_curve = equal_count_bin_medians(x_data, y_data)
            if len(bin_curve) >= 2:
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='white', linewidth=4.2, zorder=3,
                )
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='#20252B', linewidth=2.0, marker='o',
                    markersize=3.4, markerfacecolor='white',
                    markeredgecolor='#20252B', markeredgewidth=0.8,
                    zorder=4,
                )

            version_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == version
            ]
            raw_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == 'raw'
            ]
            if len(version_rows) == 1:
                result_row = version_rows.iloc[0]
                raw_row = raw_rows.iloc[0] if len(raw_rows) == 1 else None
                draw_power_fit(ax, result_row, x_data)
                class_name, details = format_panel_annotation(
                    result_row, version, raw_row=raw_row
                )
            else:
                class_name, details = 'No result', ''
            draw_class_annotation(ax, class_name, details, fontsize=9.2)
            ax.set_xlim(payload['xlim'])
            ax.set_ylim(payload['ylim'])
            if row_i == 0:
                ax.set_title(
                    f'{domain_label}\n({domain_key})',
                    fontsize=12, fontweight='semibold', pad=8,
                )
            ax.set_xlabel(UNIT_LABELS[x_var], fontsize=10)
            ax.set_ylabel(
                UNIT_LABELS[y_var] if col_i == 0 else '',
                fontsize=10,
            )
            if col_i == 0:
                ax.text(
                    -0.24, 0.5, pair_label,
                    transform=ax.transAxes, rotation=90,
                    va='center', ha='center',
                    fontsize=13, fontweight='bold', color='#222222',
                )
            ax.text(
                0.98, 0.02, chr(ord('a') + row_i * n_cols + col_i),
                transform=ax.transAxes, va='bottom', ha='right',
                fontsize=11, fontweight='bold', color='#444444',
            )
            style_axes(ax)
            face = '#FFFFFF'
            if len(version_rows) == 1:
                face = classification_facecolor(version_rows.iloc[0]['group'])
            ax.set_facecolor(face)

    fig.suptitle(
        f"{version.upper()} per-model  ·  {run['model']}  ·  "
        f"{START_YEAR}–{END_YEAR}",
        fontsize=16, fontweight='semibold',
    )
    legend_handles = [
        Line2D(
            [0], [0], color='#20252B', linewidth=2.0, marker='o',
            markersize=4, markerfacecolor='white', alpha=0.6,
            label=f'Median trend ({N_EQUAL_COUNT_BINS} equal-count bins)',
        ),
        Line2D(
            [0], [0], color='#B12A68', linewidth=1.8,
            linestyle=(0, (5, 3)), alpha=0.6,
            label='Power-law fit (when fitted)',
        ),
        Line2D(
            [0], [0], marker='x', color='#444444', linestyle='none',
            markersize=8, alpha=0.6,
            label='Removed in trimmed (outside 1st–99th percentile)',
        ),
    ]
    for key, label in ZONE_LEGEND:
        legend_handles.append(
            Line2D(
                [0], [0], marker='o', color='none',
                markerfacecolor=ZONE_COLORS[key], markeredgecolor='none',
                markersize=8, alpha=0.6, label=label,
            )
        )
    fig.legend(
        handles=legend_handles, loc='lower center', ncol=7,
        bbox_to_anchor=(0.5, -0.06), frameon=True, fancybox=True, framealpha=0.5, edgecolor='none', fontsize=11,
    )
    if WRITE_OUTPUTS:
        fig.savefig(
            PAIR_FIGURE_DIR / f"S4_relationships_{run['model']}_{version}.png",
            dpi=180, bbox_inches='tight',
        )
    plt.show()
    plt.close(fig)


# print('Per-model RAW scatter plots')
# print('=' * 72)
# for run in RUNS:
#     payloads = build_panel_payloads(run)
#     run['panel_payloads'] = payloads
#     draw_relationship_figure(run, 'raw', payloads)


## 6d. Per-model scatter plots — TRIMMED / 单模型散点（trimmed）

Same 3 × 7 per-model layout as the raw figures above.

**中文说明：** 与上一节单模型图相同，仅在全部 raw 单模型图之后显示 trimmed。

Trimmed panels keep retained cells as circles and mark removed cells with **x**.

**中文说明：** trimmed 图里保留的点仍是圆点，被去掉的点用 x。


In [9]:
# print('Per-model TRIMMED scatter plots')
# print('=' * 72)
# for run in RUNS:
#     payloads = run.get('panel_payloads')
#     if payloads is None:
#         payloads = build_panel_payloads(run)
#         run['panel_payloads'] = payloads
#     draw_relationship_figure(run, 'trimmed', payloads)


## 6e. Model × zone grid — RAW / 模型×分区总图（raw）

One figure per variable pair. **Rows are models, columns are domains.** Each panel is that model's cells in that zone. Empty domains (no ice field, etc.) stay blank.

**中文说明：** 每个变量对一张图。一行一个模型，一列一个分区。
Models with no cells in a domain are left out of this overview (e.g. no `sftgif`, so WW–CD and ice are empty).

**中文说明：** 总图不画空模型：没有该分区格点的模型整行去掉（如缺冰盖场、四区为空的那些）。


In [10]:
def _ensure_payloads(run):
    payloads = run.get('panel_payloads')
    if payloads is None:
        payloads = build_panel_payloads(run)
        run['panel_payloads'] = payloads
    return payloads


def _raw_count(run, pair_label, domain_key):
    payload = _ensure_payloads(run).get((pair_label, domain_key))
    if payload is None:
        return 0
    return len(payload['prepared']['raw'][0])


def column_limits_for_pair(pair_label, domain_key, runs, version='raw'):
    xs, ys = [], []
    for run in runs:
        payload = _ensure_payloads(run).get((pair_label, domain_key))
        if payload is None:
            continue
        xv, yv = payload['prepared'][version]
        if len(xv) == 0:
            continue
        xs.append(xv)
        ys.append(yv)
    if not xs:
        return (-1.0, 1.0), (-1.0, 1.0)
    return padded_limits(np.concatenate(xs)), padded_limits(np.concatenate(ys))


def select_grid_runs(pair_label):
    """Drop models with no cells in any domain on this pair."""
    keep, skipped = [], []
    for run in RUNS:
        n_any = sum(
            _raw_count(run, pair_label, domain_key)
            for domain_key, _, _ in DOMAINS
        )
        n_not_all_land = sum(
            _raw_count(run, pair_label, domain_key)
            for domain_key, _, _ in DOMAINS
            if domain_key != 'all_land'
        )
        # Unknown-ice models only fill All land; omit them from this overview.
        if n_not_all_land == 0:
            skipped.append(run['model'])
            continue
        if n_any == 0:
            skipped.append(run['model'])
            continue
        keep.append(run)
    if skipped:
        print(
            f'  omit empty models from overview: {", ".join(skipped)}'
        )
    return keep


def select_grid_domains(pair_label, runs):
    """Drop domain columns where none of the kept models have points."""
    keep = []
    for domain in DOMAINS:
        if any(_raw_count(run, pair_label, domain[0]) > 0 for run in runs):
            keep.append(domain)
    return keep


def draw_model_zone_grid(x_var, y_var, pair_label, version):
    grid_runs = select_grid_runs(pair_label)
    grid_domains = select_grid_domains(pair_label, grid_runs)
    n_rows = len(grid_runs)
    n_cols = len(grid_domains)
    if n_rows == 0 or n_cols == 0:
        print(f'No models with data for {pair_label} {version}')
        return

    col_limits = {
        domain_key: column_limits_for_pair(pair_label, domain_key, grid_runs, version)
        for domain_key, _, _ in grid_domains
    }

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.2 * n_cols, 2.3 * n_rows),
        constrained_layout=True,
        squeeze=False,
    )

    for row_i, run in enumerate(grid_runs):
        payloads = _ensure_payloads(run)

        for col_i, (domain_key, domain_label, _) in enumerate(grid_domains):
            ax = axes[row_i, col_i]
            payload = payloads[(pair_label, domain_key)]
            x_data, y_data = payload['prepared'][version]
            groups = payload['prepared'][f'{version}_groups']
            scatter_points(ax, x_data, y_data, domain_key, groups)

            bin_curve = equal_count_bin_medians(x_data, y_data)
            if len(bin_curve) >= 2:
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='white', linewidth=3.6, zorder=3,
                )
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='#20252B', linewidth=1.7, marker='o',
                    markersize=2.8, markerfacecolor='white',
                    markeredgecolor='#20252B', markeredgewidth=0.7,
                    zorder=4,
                )

            version_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == version
            ]
            raw_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == 'raw'
            ]
            if len(version_rows) == 1:
                result_row = version_rows.iloc[0]
                raw_row = raw_rows.iloc[0] if len(raw_rows) == 1 else None
                if version == 'trimmed':
                    x_raw, y_raw = payload['prepared']['raw']
                    local_raw_n = len(x_raw)
                    local_trim_n = len(x_data)
                    local_rem = local_raw_n - local_trim_n
                    local_frac = local_rem / local_raw_n if local_raw_n > 0 else 0.0
                    result_row = result_row.copy()
                    result_row['n_cells'] = local_trim_n
                    result_row['raw_n_cells'] = local_raw_n
                    result_row['n_removed'] = local_rem
                    result_row['removed_fraction'] = local_frac
                draw_power_fit(ax, result_row, x_data)
                class_name, details = format_panel_annotation(
                    result_row, version, raw_row=raw_row
                )
            else:
                class_name, details = 'No result', ''
            draw_class_annotation(ax, class_name, details, fontsize=8.4)

            xlim, ylim = col_limits[domain_key]
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            style_axes(ax)
            face = '#FFFFFF'
            if len(version_rows) == 1:
                face = classification_facecolor(version_rows.iloc[0]['group'])
            ax.set_facecolor(face)

            if row_i == 0:
                ax.set_title(
                    f'{domain_label}\n({domain_key})',
                    fontsize=12, fontweight='semibold', pad=6,
                )
            if col_i == 0:
                ax.set_ylabel(UNIT_LABELS[y_var], fontsize=9)
                ax.text(
                    -0.42, 0.5, run['model'],
                    transform=ax.transAxes, rotation=90,
                    va='center', ha='center',
                    fontsize=12, fontweight='bold', color='#222222',
                )
            if row_i == n_rows - 1:
                ax.set_xlabel(UNIT_LABELS[x_var], fontsize=9)

    fig.suptitle(
        f'{version.upper()}  ·  {PAIR_FULL_NAMES.get(pair_label, pair_label)}  ·  rows = models, columns = zones',
        fontsize=17, fontweight='semibold',
    )
    legend_handles = [
        Line2D(
            [0], [0], color='#20252B', linewidth=1.7, marker='o',
            markersize=4, markerfacecolor='white', alpha=0.6,
            label='Median trend',
        ),
        Line2D(
            [0], [0], color='#B12A68', linewidth=1.6,
            linestyle=(0, (5, 3)), alpha=0.6,
            label='Power-law fit (when fitted)',
        ),
    ]
    for key, label in ZONE_LEGEND:
        legend_handles.append(
            Line2D(
                [0], [0], marker='o', color='none',
                markerfacecolor=ZONE_COLORS[key], markeredgecolor='none',
                markersize=8, alpha=0.6, label=label,
            )
        )

    fig.legend(
        handles=legend_handles, loc='lower center',
        ncol=min(8, len(legend_handles)),
        bbox_to_anchor=(0.5, -0.06), frameon=True, fancybox=True, framealpha=0.5, edgecolor='none', fontsize=11,
    )
    if WRITE_OUTPUTS:
        path = PAIR_FIGURE_DIR / (
            f'S4_modelgrid_{PAIR_SLUGS[pair_label]}_{version}.png'
        )
        fig.savefig(path, dpi=160, bbox_inches='tight')
        print(f'saved {path}')
    plt.show()
    plt.close(fig)


# Hexbin companion figures use the same panel data and the branch result
# already produced by the decision tree; no detector is run a second time.
import kde_branch_comparison as _branch_hexbin_plotter
_branch_hexbin_plotter = importlib.reload(_branch_hexbin_plotter)


def build_branch_hexbin_inputs(pair_label, version):
    cases = {}
    objects = {}
    for run in RUNS:
        payloads = _ensure_payloads(run)
        for domain_key, _, _ in DOMAINS:
            payload = payloads.get((pair_label, domain_key))
            if payload is None:
                continue
            x_data, y_data = payload['prepared'][version]
            key = (run['model'], domain_key, pair_label, version)
            cases[key] = (x_data, y_data)
            rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == version
            ]
            if len(rows) == 1:
                row = rows.iloc[0]
                branch_status = row.get('branch_status', None)
                if row.get('gate1_gate', None) == 'no_global':
                    status = 'No global'
                elif branch_status in {'Branch', 'Candidate'}:
                    status = branch_status
                else:
                    status = 'No branch'
                mic = float(row.get('mic', np.nan))
                dcor = float(row.get('dcor', np.nan))
            else:
                status, mic, dcor = 'No global', np.nan, np.nan
            objects[key] = {
                'status': status,
                'branch_behind_gate': False,
                'mic': mic,
                'dcor': dcor,
                'mic_pass': bool(
                    np.isfinite(mic)
                    and mic >= CLASSIFIER_DEFAULTS['mic_intermediate']
                ),
                'dcor_pass': bool(
                    np.isfinite(dcor)
                    and dcor >= CLASSIFIER_DEFAULTS['dcor_intermediate']
                ),
            }
    return cases, objects


def draw_branch_hexbin_grid(pair_label, version):
    cases, objects = build_branch_hexbin_inputs(pair_label, version)
    models = [run['model'] for run in RUNS]
    domains = [domain_key for domain_key, _, _ in DOMAINS]
    expected = {
        (model, domain, pair_label, version)
        for model in models for domain in domains
    }
    missing = expected - set(cases)
    if missing:
        print(f'Skip Hexbin {pair_label} {version}: {len(missing)} missing panels')
        return None
    _branch_hexbin_plotter.MODELS = models
    _branch_hexbin_plotter.DOMAINS = domains
    _branch_hexbin_plotter.DOMAIN_LABELS.update({
        key: label for key, label, _ in DOMAINS
    })
    return _branch_hexbin_plotter.draw_hexbin_grid(
        cases, objects, pair_label, version, PAIR_FIGURE_DIR,
        PAIR_SLUGS[pair_label], show=True, gridsize=26,
    )


print('Model × zone grids — RAW')
print('=' * 72)
for x_var, y_var, pair_label in VARIABLE_PAIRS:
    print(f'\nRAW grid · {pair_label}')
    draw_model_zone_grid(x_var, y_var, pair_label, 'raw')
    draw_branch_hexbin_grid(pair_label, 'raw')


Model × zone grids — RAW

RAW grid · P → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_P_Q_raw.png

RAW grid · ET → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_ET_Q_raw.png

RAW grid · hfls → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_hfls_Q_raw.png

RAW grid · hfss → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_hfss_Q_raw.png

RAW grid · tran → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_tran_Q_raw.png

RAW grid · evspsblsoi → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_evspsblsoi_Q_raw.png

RAW grid · mrros → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch

## 6f. Model × zone grid — TRIMMED / 模型×分区总图（trimmed）

Same rows (models) and columns (zones) as the raw grids. Removed cells are marked with **x**.

**中文说明：** 布局与 raw 相同；被去掉的点用 x。


In [11]:
print('Model × zone grids — TRIMMED')
print('=' * 72)
for x_var, y_var, pair_label in VARIABLE_PAIRS:
    print(f'\nTRIMMED grid · {pair_label}')
    draw_model_zone_grid(x_var, y_var, pair_label, 'trimmed')
    draw_branch_hexbin_grid(pair_label, 'trimmed')


Model × zone grids — TRIMMED

TRIMMED grid · P → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_P_Q_trimmed.png

TRIMMED grid · ET → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_ET_Q_trimmed.png

TRIMMED grid · hfls → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_hfls_Q_trimmed.png

TRIMMED grid · hfss → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_hfss_Q_trimmed.png

TRIMMED grid · tran → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_tran_Q_trimmed.png

TRIMMED grid · evspsblsoi → Q
saved /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S4_branch/S4_modelgrid_evspsblsoi_Q_trimmed.png

TRIMMED grid · mrros → Q
saved /Users/mimi/Documents/Code/Gith

## 6g. Model × zone grid — GLOBAL TRIMMED — disabled / 全局去极值总图（已停用）

Same layout as 6f, but extreme values are removed using **global (all_land) quantile bounds** per model×pair, then the trimmed points only are shown with axis limits fitted to the remaining data. No dropped-point markers.

In [12]:
%%script --no-raise-error false
# Disabled: do not calculate or draw the separate GLOBAL TRIMMED grids.
# Source is retained below for reference.
def build_global_trim_bounds(run):
    """Compute 1-99% quantile bounds from all_land for each variable pair."""
    t = run['clim']
    bounds = {}
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        if x_var not in t.columns or y_var not in t.columns:
            continue
        x_all = np.asarray(t[x_var], dtype=float)
        y_all = np.asarray(t[y_var], dtype=float)
        finite = np.isfinite(x_all) & np.isfinite(y_all)
        xf, yf = x_all[finite], y_all[finite]
        if len(xf) == 0:
            bounds[(x_var, y_var)] = (np.nan, np.nan, np.nan, np.nan)
        else:
            x_lo, x_hi = np.quantile(xf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
            y_lo, y_hi = np.quantile(yf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
            bounds[(x_var, y_var)] = (float(x_lo), float(x_hi), float(y_lo), float(y_hi))
    return bounds


def build_panel_payloads_global(run):
    """Like build_panel_payloads but uses all_land trim bounds for every domain."""
    t = run['clim']
    global_bounds = build_global_trim_bounds(run)
    payloads = {}
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        if (x_var, y_var) not in global_bounds:
            continue
        gb = global_bounds[(x_var, y_var)]
        x_lo, x_hi, y_lo, y_hi = gb
        for domain_key, domain_label, domain_filter in DOMAINS:
            subset = t.loc[domain_filter(t)] if domain_filter is not None else t
            x_arr = np.asarray(subset[x_var], dtype=float)
            y_arr = np.asarray(subset[y_var], dtype=float)
            groups = np.asarray(subset['analysis_zone']) if 'analysis_zone' in subset else None
            finite = np.isfinite(x_arr) & np.isfinite(y_arr)
            x_raw = x_arr[finite]
            y_raw = y_arr[finite]
            g_raw = groups[finite] if groups is not None else None
            if len(x_raw) == 0 or np.isnan(x_lo):
                keep = np.zeros(len(x_raw), dtype=bool)
            else:
                keep = (
                    (x_raw >= x_lo) & (x_raw <= x_hi)
                    & (y_raw >= y_lo) & (y_raw <= y_hi)
                )
            x_trim = x_raw[keep]
            y_trim = y_raw[keep]
            g_trim = g_raw[keep] if g_raw is not None else None
            prepared = {
                'raw': (x_raw, y_raw),
                'trimmed': (x_trim, y_trim),
                'raw_groups': g_raw,
                'trimmed_groups': g_trim,
                'trim_keep': keep,
                'bounds': gb,
            }
            result_rows = RESULTS_DF.loc[
                (RESULTS_DF['model'] == run['model'])
                & (RESULTS_DF['experiment'] == run['experiment'])
                & (RESULTS_DF['member'] == run['member'])
                & (RESULTS_DF['grid'] == run['grid'])
                & (RESULTS_DF['domain'] == domain_key)
                & (RESULTS_DF['sample'] == 'all')
                & (RESULTS_DF['x_var'] == x_var)
                & (RESULTS_DF['y_var'] == y_var)
            ]
            payloads[(pair_label, domain_key)] = {
                'x_var': x_var,
                'y_var': y_var,
                'pair_label': pair_label,
                'domain_key': domain_key,
                'domain_label': domain_label,
                'prepared': prepared,
                'result_rows': result_rows,
                'xlim': padded_limits(x_trim),
                'ylim': padded_limits(y_trim),
            }
    return payloads


def draw_model_zone_grid_global(x_var, y_var, pair_label):
    """Model x zone grid using global trim bounds, showing only kept points."""
    grid_runs = select_grid_runs(pair_label)
    grid_domains = select_grid_domains(pair_label, grid_runs)
    n_rows = len(grid_runs)
    n_cols = len(grid_domains)
    if n_rows == 0 or n_cols == 0:
        print(f'No models with data for {pair_label} global-trimmed')
        return

    all_payloads = {}
    for run in grid_runs:
        all_payloads[run['model']] = build_panel_payloads_global(run)

    col_limits = {}
    for domain_key, _, _ in grid_domains:
        xs, ys = [], []
        for run in grid_runs:
            p = all_payloads[run['model']].get((pair_label, domain_key))
            if p is None:
                continue
            xt, yt = p['prepared']['trimmed']
            if len(xt) > 0:
                xs.append(xt)
                ys.append(yt)
        if xs:
            col_limits[domain_key] = (
                padded_limits(np.concatenate(xs)),
                padded_limits(np.concatenate(ys)),
            )
        else:
            col_limits[domain_key] = ((-1.0, 1.0), (-1.0, 1.0))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.2 * n_cols, 2.3 * n_rows),
        constrained_layout=True,
        squeeze=False,
    )

    for row_i, run in enumerate(grid_runs):
        payloads = all_payloads[run['model']]
        for col_i, (domain_key, domain_label, _) in enumerate(grid_domains):
            ax = axes[row_i, col_i]
            payload = payloads.get((pair_label, domain_key))
            if payload is None:
                ax.text(
                    0.5, 0.5, 'Unavailable', transform=ax.transAxes,
                    ha='center', va='center', fontsize=12, color='#6B7280',
                )
                style_axes(ax)
                continue
            x_data, y_data = payload['prepared']['trimmed']
            groups = payload['prepared']['trimmed_groups']
            scatter_points(ax, x_data, y_data, domain_key, groups)

            bin_curve = equal_count_bin_medians(x_data, y_data)
            if len(bin_curve) >= 2:
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='white', linewidth=3.6, zorder=3,
                )
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='#20252B', linewidth=1.7, marker='o',
                    markersize=2.8, markerfacecolor='white',
                    markeredgecolor='#20252B', markeredgewidth=0.7,
                    zorder=4,
                )

            version_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == 'trimmed'
            ]
            raw_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == 'raw'
            ]
            if len(version_rows) == 1:
                result_row = version_rows.iloc[0]
                raw_row = raw_rows.iloc[0] if len(raw_rows) == 1 else None
                draw_power_fit(ax, result_row, x_data)
                class_name, details = format_panel_annotation(
                    result_row, 'trimmed', raw_row=raw_row
                )
            else:
                class_name, details = 'No result', ''
            draw_class_annotation(ax, class_name, details, fontsize=8.4)

            xlim, ylim = col_limits[domain_key]
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            style_axes(ax)
            face = '#FFFFFF'
            if len(version_rows) == 1:
                face = classification_facecolor(version_rows.iloc[0]['group'])
            ax.set_facecolor(face)

            if row_i == 0:
                ax.set_title(
                    f'{domain_label}\n({domain_key})',
                    fontsize=12, fontweight='semibold', pad=6,
                )
            if col_i == 0:
                ax.set_ylabel(UNIT_LABELS[y_var], fontsize=9)
                ax.text(
                    -0.42, 0.5, run['model'],
                    transform=ax.transAxes, rotation=90,
                    va='center', ha='center',
                    fontsize=12, fontweight='bold', color='#222222',
                )
            if row_i == n_rows - 1:
                ax.set_xlabel(UNIT_LABELS[x_var], fontsize=9)

    fig.suptitle(
        f'GLOBAL TRIMMED  ·  {PAIR_FULL_NAMES.get(pair_label, pair_label)}  ·  rows = models, columns = zones',
        fontsize=17, fontweight='semibold',
    )
    legend_handles = [
        Line2D(
            [0], [0], color='#20252B', linewidth=1.7, marker='o',
            markersize=4, markerfacecolor='white', alpha=0.6,
            label='Median trend',
        ),
        Line2D(
            [0], [0], color='#B12A68', linewidth=1.6,
            linestyle=(0, (5, 3)), alpha=0.6,
            label='Power-law fit (when fitted)',
        ),
    ]
    for key, label in ZONE_LEGEND:
        legend_handles.append(
            Line2D(
                [0], [0], marker='o', color='none',
                markerfacecolor=ZONE_COLORS[key], markeredgecolor='none',
                markersize=8, alpha=0.6, label=label,
            )
        )
    fig.legend(
        handles=legend_handles, loc='lower center',
        ncol=min(7, len(legend_handles)),
        bbox_to_anchor=(0.5, -0.06), frameon=True, fancybox=True, framealpha=0.5, edgecolor='none', fontsize=11,
    )
    if WRITE_OUTPUTS:
        path = PAIR_FIGURE_DIR / (
            f'S4_modelgrid_{PAIR_SLUGS[pair_label]}_global_trimmed.png'
        )
        fig.savefig(path, dpi=160, bbox_inches='tight')
        print(f'saved {path}')
    plt.show()
    plt.close(fig)


print('Model × zone grids — GLOBAL TRIMMED')
print('=' * 72)
for x_var, y_var, pair_label in VARIABLE_PAIRS:
    print(f'\nGLOBAL TRIMMED grid · {pair_label}')
    draw_model_zone_grid_global(x_var, y_var, pair_label)


## 7. Classification heatmap — RAW / raw分类热力图

One heatmap per model for **raw** classifications (domain × variable pair). Trimmed heatmaps follow in the next section.

**中文说明：** 每个模型先画一张raw热力图；全部raw热力图完成后再画trimmed。


In [13]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# domain_order = [d[0] for d in DOMAINS]
# pair_order = [v[2] for v in VARIABLE_PAIRS]
# GROUP_COLORS = {
#     'Linear': '#4CAF50',
#     'Saturation': '#FF9800',
#     'Acceleration': '#F44336',
#     'U-shape': '#9C27B0',
#     'Cubic': '#3F51B5',
#     'Oscillation': '#00BCD4',
#     'Branch': '#795548',
#     'Transition': '#607D8B',
#     'No Global Relationship': '#BDBDBD',
#     'Uncertain': '#E0E0E0',
#     'Insufficient data': '#F3F4F6',
# }
# 
# 
# def draw_classification_heatmap(ax, result_table, version):
#     version_table = result_table.loc[
#         (result_table['sample'] == 'all')
#         & (result_table['outlier_version'] == version)
#     ]
#     pivot_group = version_table.pivot_table(
#         index='domain', columns='pair', values='group', aggfunc='first',
#     ).reindex(index=domain_order, columns=pair_order)
#     pivot_mic = version_table.pivot_table(
#         index='domain', columns='pair', values='mic', aggfunc='first',
#     ).reindex(index=domain_order, columns=pair_order)
# 
#     ax.set_xlim(-0.5, len(pair_order) - 0.5)
#     ax.set_ylim(-0.5, len(domain_order) - 0.5)
#     ax.invert_yaxis()
#     ax.set_xticks(range(len(pair_order)))
#     ax.set_xticklabels(pair_order)
#     ax.set_yticks(range(len(domain_order)))
#     ax.set_yticklabels([d[1] for d in DOMAINS])
#     for yi, domain in enumerate(domain_order):
#         for xi, pair in enumerate(pair_order):
#             group_value = pivot_group.loc[domain, pair]
#             group = group_value if pd.notna(group_value) else ''
#             mic_val = pivot_mic.loc[domain, pair]
#             rect = plt.Rectangle(
#                 (xi - 0.48, yi - 0.45), 0.96, 0.9,
#                 facecolor=GROUP_COLORS.get(group, '#F5F5F5'),
#                 edgecolor='white', linewidth=2,
#             )
#             ax.add_patch(rect)
#             ax.text(xi, yi - 0.1, group, ha='center', va='center',
#                     fontsize=11, fontweight='semibold', color='#222222')
#             if pd.notna(mic_val):
#                 ax.text(xi, yi + 0.2, f'MIC={mic_val:.3f}',
#                         ha='center', va='center', fontsize=10, color='#555555')
#     ax.set_title(f'{version.upper()} classification',
#                  fontsize=14, fontweight='semibold', pad=10)
#     style_axes(ax)
#     ax.grid(False)
# 
# 
# def draw_heatmap_figure(run, version):
#     run_results = RESULTS_DF.loc[
#         (RESULTS_DF['model'] == run['model'])
#         & (RESULTS_DF['experiment'] == run['experiment'])
#         & (RESULTS_DF['member'] == run['member'])
#         & (RESULTS_DF['grid'] == run['grid'])
#     ]
#     fig, ax = plt.subplots(figsize=(10.5, 6.8), constrained_layout=True)
#     draw_classification_heatmap(ax, run_results, version)
#     fig.suptitle(
#         f"{version.upper()} spatial classifications · {run['model']}",
#         fontsize=14, fontweight='semibold',
#     )
#     if WRITE_OUTPUTS:
#         fig.savefig(
#             PAIR_FIGURE_DIR / f"S4_classification_heatmap_{run['model']}_{version}.png",
#             bbox_inches='tight',
#         )
#     plt.show()
#     plt.close(fig)
# 
# 
# print('RAW classification heatmaps')
# print('=' * 72)
# for run in RUNS:
#     draw_heatmap_figure(run, 'raw')
# 

## 7b. Classification heatmap — TRIMMED / trimmed分类热力图

Same domain × pair layout as the raw heatmaps, shown only after every model's raw heatmap.

**中文说明：** 与raw热力图布局相同，全部raw热力图完成后再显示。


In [14]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# print('TRIMMED classification heatmaps')
# print('=' * 72)
# for run in RUNS:
#     draw_heatmap_figure(run, 'trimmed')
# 

## 8. Sensitivity analysis / 敏感性分析

Two independent sensitivity checks are reported. First, all versus core is compared separately for raw and trimmed data to diagnose boundary mixing. Second, raw versus trimmed is compared for every domain × sample × pair to diagnose dependence on extreme grid cells. Raw remains the primary result.

**中文说明：** 这里分开检查两种敏感性：(1) 在raw和trimmed内部各自比较all与core，用于判断分区边界混合；(2) 对每个domain × sample × pair比较raw与trimmed，用于判断分类是否依赖极端格点。raw始终是主结果。


In [15]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# RUN_KEYS = ['model', 'experiment', 'member', 'grid']
# zone_domains = ['WW', 'WD', 'CW', 'CD']
# 
# # A. all versus core, evaluated separately inside raw and trimmed.
# core_sensitivity_rows = []
# for run_key, run_cases in RESULTS_DF.groupby(RUN_KEYS, sort=False):
#     run_meta = dict(zip(RUN_KEYS, run_key))
#     for version in ['raw', 'trimmed']:
#         version_cases = run_cases.loc[run_cases['outlier_version'] == version]
#         for domain in zone_domains:
#             for x_var, y_var, pair_label in VARIABLE_PAIRS:
#                 common = (
#                     (version_cases['domain'] == domain)
#                     & (version_cases['x_var'] == x_var)
#                     & (version_cases['y_var'] == y_var)
#                 )
#                 all_row = version_cases.loc[common & version_cases['sample'].eq('all')]
#                 core_row = version_cases.loc[common & version_cases['sample'].eq('core')]
#                 if len(all_row) != 1 or len(core_row) != 1:
#                     continue
#                 a = all_row.iloc[0]
#                 c = core_row.iloc[0]
#                 core_sensitivity_rows.append({
#                     **run_meta,
#                     'outlier_version': version,
#                     'domain': domain,
#                     'pair': pair_label,
#                     'all_n': a['n_cells'],
#                     'all_group': a['group'],
#                     'all_mic': a['mic'],
#                     'core_n': c['n_cells'],
#                     'core_group': c['group'],
#                     'core_mic': c['mic'],
#                     'consistent': a['group'] == c['group'],
#                 })
# 
# core_sensitivity_df = pd.DataFrame(core_sensitivity_rows)
# print('\n=== A. All vs core sensitivity ===')
# for _, _, pair_label in VARIABLE_PAIRS:
#     print(f'\n--- {pair_label} ---')
#     display(core_sensitivity_df.loc[
#         core_sensitivity_df['pair'] == pair_label,
#         ['domain', 'outlier_version', 'all_n', 'all_group', 'all_mic',
#          'core_n', 'core_group', 'core_mic', 'consistent'],
#     ].round(4))
# 
# # B. raw versus trimmed for every exact domain × sample × pair.
# outlier_sensitivity_rows = []
# unit_cols = RUN_KEYS + ['domain', 'sample', 'x_var', 'y_var', 'pair']
# for unit_key, unit_cases in RESULTS_DF.groupby(unit_cols, sort=False):
#     unit_meta = dict(zip(unit_cols, unit_key))
#     raw_row = unit_cases.loc[unit_cases['outlier_version'] == 'raw']
#     trimmed_row = unit_cases.loc[unit_cases['outlier_version'] == 'trimmed']
#     if len(raw_row) != 1 or len(trimmed_row) != 1:
#         continue
#     raw = raw_row.iloc[0]
#     trimmed = trimmed_row.iloc[0]
#     outlier_sensitivity_rows.append({
#         **unit_meta,
#         'raw_n': raw['n_cells'],
#         'trimmed_n': trimmed['n_cells'],
#         'n_removed': trimmed['n_removed'],
#         'removed_fraction': trimmed['removed_fraction'],
#         'raw_mic': raw.get('mic', np.nan),
#         'trimmed_mic': trimmed.get('mic', np.nan),
#         'raw_pearson': raw.get('pearson_r_descriptive', np.nan),
#         'trimmed_pearson': trimmed.get('pearson_r_descriptive', np.nan),
#         'raw_power_r2': raw.get('power_r2', np.nan),
#         'trimmed_power_r2': trimmed.get('power_r2', np.nan),
#         'raw_power_b': raw.get('power_b', np.nan),
#         'trimmed_power_b': trimmed.get('power_b', np.nan),
#         'raw_group': raw['group'],
#         'trimmed_group': trimmed['group'],
#         'consistent': raw['group'] == trimmed['group'],
#     })
# 
# outlier_sensitivity_df = pd.DataFrame(outlier_sensitivity_rows)
# print('\n=== B. Raw vs trimmed outlier sensitivity ===')
# outlier_display_cols = [
#     'domain', 'sample', 'raw_n', 'trimmed_n', 'n_removed',
#     'removed_fraction', 'raw_mic', 'trimmed_mic',
#     'raw_pearson', 'trimmed_pearson',
#     'raw_power_r2', 'trimmed_power_r2',
#     'raw_power_b', 'trimmed_power_b',
#     'raw_group', 'trimmed_group', 'consistent',
# ]
# for _, _, pair_label in VARIABLE_PAIRS:
#     print(f'\n--- {pair_label} ---')
#     display(outlier_sensitivity_df.loc[
#         outlier_sensitivity_df['pair'] == pair_label,
#         outlier_display_cols,
#     ].round(4))
# 
# n_changed = int((~outlier_sensitivity_df['consistent']).sum())
# print(f'\nRaw/trimmed classification changes: {n_changed} / {len(outlier_sensitivity_df)}')
# if n_changed > 0:
#     display(outlier_sensitivity_df.loc[
#         ~outlier_sensitivity_df['consistent'],
#         ['domain', 'sample', 'pair', 'raw_group', 'trimmed_group',
#          'removed_fraction'],
#     ].round(4))

## 9. Save results / 保存结果

Save the full raw/trimmed classification table, the all/core comparison, and the raw/trimmed outlier-sensitivity comparison.

**中文说明：** 保存包含raw/trimmed的完整分类结果表、all/core敏感性表和raw/trimmed极值敏感性表。


In [16]:
# for run in RUNS:
#     output_dir = run['zones_dir']
#     period = f'{START_YEAR}_{END_YEAR}'

#     if not WRITE_OUTPUTS:
#         print(f"WRITE_OUTPUTS=False, skipping save for {run['model']}")
#         continue

#     run_results = RESULTS_DF.loc[
#         (RESULTS_DF['model'] == run['model'])
#         & (RESULTS_DF['experiment'] == run['experiment'])
#         & (RESULTS_DF['member'] == run['member'])
#         & (RESULTS_DF['grid'] == run['grid'])
#     ].copy()

#     results_path = output_dir / f'S4_classification_results_{period}.csv'
#     run_results.to_csv(results_path, index=False)

#     core_sens_path = output_dir / f'S4_core_sensitivity_{period}.csv'
#     run_core_sensitivity = core_sensitivity_df.loc[
#         (core_sensitivity_df['model'] == run['model'])
#         & (core_sensitivity_df['experiment'] == run['experiment'])
#         & (core_sensitivity_df['member'] == run['member'])
#         & (core_sensitivity_df['grid'] == run['grid'])
#     ]
#     run_core_sensitivity.to_csv(core_sens_path, index=False)

#     outlier_sens_path = output_dir / f'S4_outlier_sensitivity_{period}.csv'
#     run_outlier_sensitivity = outlier_sensitivity_df.loc[
#         (outlier_sensitivity_df['model'] == run['model'])
#         & (outlier_sensitivity_df['experiment'] == run['experiment'])
#         & (outlier_sensitivity_df['member'] == run['member'])
#         & (outlier_sensitivity_df['grid'] == run['grid'])
#     ]
#     run_outlier_sensitivity.to_csv(outlier_sens_path, index=False)

#     outputs = [results_path, core_sens_path, outlier_sens_path]
#     figures = sorted(output_dir.glob('S4_*.png'))
#     all_outputs = outputs + figures
#     manifest = pd.DataFrame([{
#         'product': p.name,
#         'size_MB': p.stat().st_size / 1e6,
#     } for p in all_outputs])
#     display(manifest.round({'size_MB': 3}))

# pair_figs = sorted(PAIR_FIGURE_DIR.glob('S4_*')) if 'PAIR_FIGURE_DIR' in globals() else []
# if pair_figs:
#     print('\nPair-level summaries:')
#     display(pd.DataFrame([
#         {'product': f.name, 'size_MB': f.stat().st_size / 1e6}
#         for f in pair_figs
#     ]).round({'size_MB': 3}))
# print('S4 completed successfully.')
